<a href="https://colab.research.google.com/github/Hajorda/fireye/blob/main/notebooks/01_dataset_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FireEye — Dataset Preparation

Downloads all 4 source datasets, normalizes class IDs, merges, augments, and uploads to HuggingFace.

**Run time**: ~3–5 hours on Colab (mostly download + augmentation)

**Before running:**
1. Add `HF_TOKEN` to Colab Secrets (key icon in left sidebar)
2. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab Secrets
3. Make sure Google Drive has 25+ GB free

In [1]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_ROOT = '/content/drive/MyDrive/fireye'
os.makedirs(REPO_ROOT, exist_ok=True)
os.chdir(REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

Mounted at /content/drive
Working directory: /content/drive/MyDrive/fireye


In [2]:
# ── Cell 2: Clone repo and install deps ─────────────────────────────────────
import os

if not os.path.exists('/content/drive/MyDrive/fireye/scripts'):
    !git clone https://github.com/hajorda/fireye.git /content/drive/MyDrive/fireye
else:
    print('Repo already cloned.')

!pip install -q ultralytics albumentations datasets huggingface_hub imagehash kaggle tqdm opencv-python-headless Pillow

Cloning into '/content/drive/MyDrive/fireye'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 23 (delta 2), reused 23 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 21.37 KiB | 1.94 MiB/s, done.
Resolving deltas: 100% (2/2), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 32.7 MB/s eta 0:00:00


In [3]:
# ── Cell 3: Set up credentials from Colab Secrets ───────────────────────────
import os
from google.colab import userdata

# HuggingFace token
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Kaggle credentials
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Write kaggle.json so the kaggle CLI works
import json
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({'username': os.environ['KAGGLE_USERNAME'], 'key': os.environ['KAGGLE_KEY']}, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
print('Credentials configured.')

Credentials configured.


In [4]:
# ── Cell 4: Download D-Fire (Kaggle) ────────────────────────────────────────
# ~1.2 GB download
!python scripts/download_datasets.py --root data/raw --datasets dfire


[1/4] Downloading D-Fire from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/sayedgamal99/smoke-fire-detection-yolo
License(s): CC0-1.0
100% 2.84G/2.84G [01:15<00:00, 40.4MB/s]

  D-Fire saved to data/raw/dfire

All downloads complete.


In [5]:
# ── Cell 5: Download Pyro-SDIS (HuggingFace) ────────────────────────────────
# ~4 GB — serializes to disk as images + YOLO .txt labels
!python scripts/download_datasets.py --root data/raw --datasets pyro_sdis


[2/4] Downloading Pyro-SDIS from HuggingFace...
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'pyronear/pyro-sdis' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
README.md: 7.37kB [00:00, 3.79MB/s]
data/train-00000-of-00006.parquet: 100% 481M/481M [00:03<00:00, 140MB/s]
data/train-00001-of-00006.parquet: 100% 485M/485M [00:02<00:00, 201MB/s]
data/train-00002-of-00006.parquet: 100% 482M/482M [00:04<00:00, 120MB/s]
data/train-00003-of-00006.parquet: 100% 483M/483M [00:06<00:00, 70.9MB/s]
data/train-00004-of-00006.parquet: 100% 480M/480M [00:03<00:00, 149MB/s]
data/train-00005-of-00006.parquet: 100% 483M/483M [00:03<00:00, 134MB/s]
data/val-00000-of-00001.parquet: 100% 390M/390M [00:02<00:00, 139MB/s]
Generating train split: 100% 29537/29537 [00:16<00:00, 1831.26 examples/s]
Generating val spl

In [6]:

# ── Cell 6: Download AI for Mankind (Google Drive) + convert VOC → YOLO ──────
import os, shutil
from pathlib import Path
from xml.etree import ElementTree as ET

!pip install -q gdown
!gdown "1sEB77bfp2yMkgsSW9703vwDHol_cK6D5" -O /tmp/aiformankind.tar.gz

os.makedirs('data/raw/aiformankind/images', exist_ok=True)
os.makedirs('data/raw/aiformankind/labels', exist_ok=True)
!tar -xzf /tmp/aiformankind.tar.gz -C data/raw/aiformankind/

src_dir = Path('data/raw/aiformankind')
img_out = src_dir / 'images'
lbl_out = src_dir / 'labels'

xml_files = list(src_dir.rglob('*.xml'))
print(f"Found {len(xml_files)} XML annotations")

converted, skipped = 0, 0
for xml_path in xml_files:
    try:
        raw = xml_path.read_bytes().lstrip(b'\xef\xbb\xbf')
        root = ET.fromstring(raw)
    except ET.ParseError:
        skipped += 1
        continue

    filename = root.findtext('filename')
    size = root.find('size')
    if filename is None or size is None:
        skipped += 1
        continue

    W = int(size.findtext('width'))
    H = int(size.findtext('height'))
    if W == 0 or H == 0:
        skipped += 1
        continue

    lines = []
    for obj in root.findall('object'):
        cls_id = 1
        bb = obj.find('bndbox')
        if bb is None:
            continue
        xmin, ymin = float(bb.findtext('xmin')), float(bb.findtext('ymin'))
        xmax, ymax = float(bb.findtext('xmax')), float(bb.findtext('ymax'))
        xc = min(max((xmin + xmax) / (2 * W), 0.0), 1.0)
        yc = min(max((ymin + ymax) / (2 * H), 0.0), 1.0)
        bw = min(max((xmax - xmin) / W,       0.0), 1.0)
        bh = min(max((ymax - ymin) / H,       0.0), 1.0)
        lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    stem = Path(filename).stem
    (lbl_out / f"{stem}.txt").write_text('\n'.join(lines))

    for ext in ('.jpg', '.jpeg', '.png'):
        matches = list(src_dir.rglob(f"{stem}{ext}"))
        if matches:
            dst = img_out / matches[0].name
            if matches[0].resolve() != dst.resolve():  # skip if already there
                shutil.copy2(matches[0], dst)
            break

    converted += 1

print(f"Converted : {converted}")
print(f"Skipped   : {skipped} (malformed XML)")
print(f"Saved to  : data/raw/aiformankind/")


Downloading...
From (original): https://drive.google.com/uc?id=1sEB77bfp2yMkgsSW9703vwDHol_cK6D5
From (redirected): https://drive.google.com/uc?id=1sEB77bfp2yMkgsSW9703vwDHol_cK6D5&confirm=t&uuid=2ec26ba3-7ca0-47fe-a514-af70e007c563
To: /tmp/aiformankind.tar.gz
100% 28.6M/28.6M [00:00<00:00, 181MB/s]
Found 1488 XML annotations
Converted : 744
Skipped   : 744 (malformed XML)
Saved to  : data/raw/aiformankind/


In [3]:
# Cell 7 replacement — FireAndSmoke via Roboflow Universe
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY")
project = rf.workspace("costicatargiu").project("firesmoke")
dataset = project.version(1).download("yolov8", location="data/raw/catargiu")

ModuleNotFoundError: No module named 'roboflow'

In [10]:
import os
os.chdir('/content/drive/MyDrive/fireye')

!python scripts/normalize_labels.py \
    --raw-root data/raw \
    --processed-root data/processed \
    --sources dfire pyro_sdis aiformankind

    # ── Cell 8: Normalize class IDs ──────────────────────────────────────────────
# Remaps all sources to fire=0, smoke=1


  dfire: 100% 21527/21527 [20:37<00:00, 17.39it/s]
  [dfire] 21527 images, 26557 annotations (remap: {0: 1, 1: 0})
  pyro_sdis: 100% 29537/29537 [35:53<00:00, 13.71it/s]
  [pyro_sdis] 29537 images, 0 annotations (remap: {0: 1})
  aiformankind: 100% 10/10 [00:00<00:00, 14.42it/s]
  [aiformankind] 10 images, 0 annotations (remap: {0: 1})

Label normalization complete.


In [11]:
# ── Cell 9: Merge + deduplicate + split ──────────────────────────────────────
# 70/20/10 stratified split; pHash deduplication (threshold=8)
!python scripts/merge_datasets.py --processed-root data/processed --merged-root data/merged

  aiformankind: 5 images
  dfire: 21527 images
  pyro_sdis: 29537 images

Deduplicating...
  Hashing dfire: 100% 21527/21527 [18:18<00:00, 19.60it/s]
  Hashing pyro_sdis: 100% 29537/29537 [36:30<00:00, 13.48it/s]
  Hashing aiformankind: 100% 5/5 [00:00<00:00, 12.28it/s]
  Deduplication: removed 35494 near-duplicate images

Splitting...
  train: 10901 images
  val: 3114 images
  test: 1560 images

Copying files...
  Copying train: 100% 10901/10901 [12:05<00:00, 15.03it/s]
  Copying val: 100% 3114/3114 [03:18<00:00, 15.72it/s]
  Copying test: 100% 1560/1560 [01:37<00:00, 15.93it/s]

Writing dataset.yaml...
  Wrote data/merged/dataset.yaml

Merge complete — 15575 total images in data/merged


In [12]:
# ── Cell 10: Offline augmentation (train split only) ─────────────────────────
# Generates 2× variants per image (4× for fire-only images)
# This is the slowest step — ~1–2 hours
!python scripts/augment_dataset.py --merged-root data/merged --multiplier 2 --fire-multiplier 4

/content/drive/MyDrive/fireye/configs/augment_config.py:25: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Augmenting train split:   0% 18/10901 [00:22<4:03:34,  1.34s/it]

KeyboardInterrupt: 

In [13]:
# ── Cell 11: Verify dataset integrity ────────────────────────────────────────
!python scripts/verify_dataset.py --merged-root data/merged

# Show the sample grid
from IPython.display import Image as IPImage
IPImage('data/merged/sample_grid.jpg', width=800)


[train]
Traceback (most recent call last):
  File "/content/drive/MyDrive/fireye/scripts/verify_dataset.py", line 193, in <module>
    main()
  File "/content/drive/MyDrive/fireye/scripts/verify_dataset.py", line 159, in main
    stats = verify_split(split, img_dir, lbl_dir)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/fireye/scripts/verify_dataset.py", line 80, in verify_split
    labels = load_yolo_labels(lbl_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/fireye/scripts/verify_dataset.py", line 35, in load_yolo_labels
    rows.append((int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])))
                 ^^^^^^^^^^^^^
ValueError: invalid literal for int() with base 10: '0.0'


FileNotFoundError: No such file or directory: 'data/merged/sample_grid.jpg'

FileNotFoundError: No such file or directory: 'data/merged/sample_grid.jpg'

<IPython.core.display.Image object>

In [15]:
# ── Cell 12: Push to HuggingFace Hub ─────────────────────────────────────────
# Creates repo: hajorda/fireye-wildfire-detection
# ~30–60 min depending on dataset size and shard size
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

!python scripts/push_to_hub.py --merged-root data/merged


Building train split...
  Building train:   6% 673/10937 [02:44<41:42,  4.10it/s]
Traceback (most recent call last):
  File "/content/drive/MyDrive/fireye/scripts/push_to_hub.py", line 157, in <module>
    main()
  File "/content/drive/MyDrive/fireye/scripts/push_to_hub.py", line 138, in main
    examples = build_examples(img_dir, lbl_dir, hf_split)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/fireye/scripts/push_to_hub.py", line 82, in build_examples
    annotations = load_yolo_labels(lbl_path)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/fireye/scripts/push_to_hub.py", line 62, in load_yolo_labels
    cls_id = int(parts[0])
             ^^^^^^^^^^^^^
ValueError: invalid literal for int() with base 10: '1.0'


## Done!

Dataset is now live at: https://huggingface.co/datasets/hajorda/fireye-wildfire-detection

Open `02_train_yolov8m.ipynb` to start training.